In [26]:
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import joblib

## 加载数据

In [27]:
# 读取Excel文件
df = pd.read_csv("/home/zhanyu/experiment/data-hh/海航系航班销售过程数据202301-202406-新/2023-01-01.csv")

In [28]:
# 查看数据的前几行
print(df.head(60))

          航班日期             A             B             C  \
0   2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
1   2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  So/c8CkA/Xs=   
2   2023-01-01  a2cXIUGpIbw=  sqfrtGIRD04=           NaN   
3   2023-01-01  sqfrtGIRD04=  a2cXIUGpIbw=           NaN   
4   2023-01-01  /M8gHzjxpNU=  gK3uAaRHrOs=           NaN   
5   2023-01-01  KETr2NAmAHE=  mU00qtw41uE=           NaN   
6   2023-01-01  mU00qtw41uE=  KETr2NAmAHE=           NaN   
7   2023-01-01  AlGb30xUGAU=  xIqe+QQIQXQ=           NaN   
8   2023-01-01  xIqe+QQIQXQ=  AlGb30xUGAU=           NaN   
9   2023-01-01  /M8gHzjxpNU=  0J4jz5aCatU=           NaN   
10  2023-01-01  /M8gHzjxpNU=  Qe7TEMbQt6o=           NaN   
11  2023-01-01  /M8gHzjxpNU=  IfQVgdMP7ik=  Qe7TEMbQt6o=   
12  2023-01-01  /M8gHzjxpNU=  IfQVgdMP7ik=  Qe7TEMbQt6o=   
13  2023-01-01  Qe7TEMbQt6o=  IfQVgdMP7ik=  /M8gHzjxpNU=   
14  2023-01-01  Qe7TEMbQt6o=  IfQVgdMP7ik=  /M8gHzjxpNU=   
15  2023-01-01  FWD6O2fHsIU=  /M8gHzjxpN

In [29]:
# 统计 flightNo 字段中不同取值的数量和出现次数
flight_counts = df['航班号'].value_counts()

# 查看结果
print(flight_counts)

航班号
A4tgQv5cqkc=    204
8mSRwscwKZM=    161
MGhRGDWjWTA=    158
gsZb3gQVFss=    152
emtiRuKmNEY=    136
               ... 
FyLlI017mcQ=      1
oSeUdhzk8GQ=      1
rfsDv6CI3fg=      1
1/ok5bqU9c8=      1
0QaPxbL9nbA=      1
Name: count, Length: 5599, dtype: int64


In [30]:
# 筛选出航班号为 'A4tgQv5cqkc=' 的行
filtered_df = df[df['航班号'] == 'A4tgQv5cqkc=']

# 查看筛选后的结果
print(filtered_df)

# 假设 df 是你的 DataFrame
filtered_df.to_csv('A4tgQv5cqkc=.csv', index=False, encoding='utf-8-sig')

              航班日期             A             B             C  \
26      2023-01-01  zwVGiYamVc0=  AlGb30xUGAU=  xIqe+QQIQXQ=   
27      2023-01-01  zwVGiYamVc0=  AlGb30xUGAU=  xIqe+QQIQXQ=   
105     2023-01-01  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
106     2023-01-01  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
107     2023-01-01  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
...            ...           ...           ...           ...   
100174  2023-01-01  zwVGiYamVc0=  AlGb30xUGAU=  xIqe+QQIQXQ=   
100175  2023-01-01  zwVGiYamVc0=  AlGb30xUGAU=  xIqe+QQIQXQ=   
100631  2023-01-01  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
100632  2023-01-01  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   
100633  2023-01-01  xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=   

                               航段           航班号  距离天数   人数  
26      AlGb30xUGAU=-xIqe+QQIQXQ=  A4tgQv5cqkc=    29   38  
27      zwVGiYamVc0=-AlGb30xUGAU=  A4tgQv5cqkc=    29   35  
105     xIqe+QQIQXQ=-t+kFcRdzL6o=  A4tgQv5cqkc= 

In [35]:
# 按 'A', 'B', 'C', '航段' 分组并统计出现次数
comb_counts = filtered_df.groupby(['A', 'B', 'C', '航段']).size()

# 将结果查看为 Series 格式
print(comb_counts)

A             B             C             航段                       
xIqe+QQIQXQ=  t+kFcRdzL6o=  /M8gHzjxpNU=  t+kFcRdzL6o=-/M8gHzjxpNU=    38
                                          xIqe+QQIQXQ=-/M8gHzjxpNU=    38
                                          xIqe+QQIQXQ=-t+kFcRdzL6o=    38
zwVGiYamVc0=  AlGb30xUGAU=  xIqe+QQIQXQ=  AlGb30xUGAU=-xIqe+QQIQXQ=    38
                                          zwVGiYamVc0=-AlGb30xUGAU=    38
                                          zwVGiYamVc0=-xIqe+QQIQXQ=    14
dtype: int64


In [31]:
# 统计 flightNo 字段中不同取值的数量和出现次数
flight_counts = filtered_df['距离天数'].value_counts()
# 查看结果
print(flight_counts)

距离天数
 0     12
 1     12
 2     12
 3     12
 4     12
 5     12
 6     11
-1      6
 13     5
 7      5
 8      5
 9      5
 10     5
 11     5
 12     5
 29     5
 28     5
 15     5
 16     5
 17     5
 18     5
 19     5
 20     5
 21     5
 22     5
 23     5
 24     5
 25     5
 26     5
 27     5
 14     5
Name: count, dtype: int64


In [33]:
# 假设 pf 是你的 DataFrame
# 统计 'A', 'B', 'C', '航段' 的组合出现次数
comb_counts = filtered_df.groupby(['A', 'B', 'C', '航段']).size().reset_index(name='count')

# 筛选出出现次数为 31 的组合
filtered_combinations = comb_counts[comb_counts['count'] == 30]

# 查看结果
print(filtered_combinations)

Empty DataFrame
Columns: [A, B, C, 航段, count]
Index: []


In [24]:
# 统计 flightNo 字段中不同取值的数量和出现次数
flight_counts = filtered_df['航段'].value_counts()
# 查看结果
print(flight_counts)

航段
AlGb30xUGAU=-xIqe+QQIQXQ=    38
zwVGiYamVc0=-AlGb30xUGAU=    38
xIqe+QQIQXQ=-t+kFcRdzL6o=    38
xIqe+QQIQXQ=-/M8gHzjxpNU=    38
t+kFcRdzL6o=-/M8gHzjxpNU=    38
zwVGiYamVc0=-xIqe+QQIQXQ=    14
Name: count, dtype: int64


In [25]:
# 统计航班号和航段字段的组合的数量和出现次数
flight_segment_counts = df.groupby(['A', 'B', 'C','航段']).size()

# 查看结果
print(flight_segment_counts)

A             B             C             航段                       
+O1iBQtlFGU=  mljW2xeLSiI=  iX2Pe2pxiqQ=  +O1iBQtlFGU=-iX2Pe2pxiqQ=     2
                                          +O1iBQtlFGU=-mljW2xeLSiI=     2
                                          mljW2xeLSiI=-iX2Pe2pxiqQ=     2
              yypkiQCX5lk=  0J4jz5aCatU=  +O1iBQtlFGU=-0J4jz5aCatU=    14
                                          +O1iBQtlFGU=-yypkiQCX5lk=    14
                                                                       ..
zwVGiYamVc0=  tKjndGSl9NQ=  xIqe+QQIQXQ=  zwVGiYamVc0=-tKjndGSl9NQ=    14
                                          zwVGiYamVc0=-xIqe+QQIQXQ=    14
              y+pJ91YLvhA=  VeRLWG2XvOc=  y+pJ91YLvhA=-VeRLWG2XvOc=    14
                                          zwVGiYamVc0=-VeRLWG2XvOc=    14
                                          zwVGiYamVc0=-y+pJ91YLvhA=    38
Length: 4008, dtype: int64
